# TripoSR · Kaggle 2×T4 Image-to-3D Worker

单图生成 GLB/OBJ · 双 T4 常驻 Worker · 从本地 Hub 鉴权拉取图片 · AES-GCM 加密回传。

Kaggle 设置：Accelerator 选择 **GPU T4 x2**，Internet 设为 **On**。推荐在 Add-ons → Secrets 中添加 `BASE_URL` 和 `KAGGLE_HUB_TOKEN`。

## 1. 安装固定版本依赖

In [ ]:
import os, sys, subprocess
from pathlib import Path

TRIPOSR_COMMIT = "107cefdc244c39106fa830359024f6a2f1c78871"
TORCHMCUBES_COMMIT = "3381600ddc3d2e4d74222f8495866be5fafbace4"
REPO = Path("/kaggle/working/TripoSR")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "setuptools>=69", "wheel"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "omegaconf==2.3.0", "einops==0.7.0", "transformers==4.35.0",
    "huggingface-hub>=0.19,<1.0", "trimesh==4.0.5", "rembg",
    "onnxruntime", "xatlas==0.0.9", "imageio[ffmpeg]",
    "aiohttp", "requests", "cryptography",
    f"git+https://github.com/tatsy/torchmcubes.git@{TORCHMCUBES_COMMIT}",
], check=True)

if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "https://github.com/VAST-AI-Research/TripoSR.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", TRIPOSR_COMMIT, "--depth=1"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", TRIPOSR_COMMIT], check=True)
sys.path.insert(0, str(REPO))
print("TripoSR source ready:", TRIPOSR_COMMIT[:12])

## 2. Hub 配置与模型下载

In [ ]:
import io, time, uuid, queue, hashlib, tempfile, threading

import numpy as np
import rembg
import requests
import torch
from PIL import Image
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from huggingface_hub import snapshot_download
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground

def secret(name, default=""):
    value = os.getenv(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name).strip()
    except Exception:
        return default

BASE = secret("BASE_URL", "https://your-tunnel.example.com").rstrip("/")
TOKEN = secret("KAGGLE_HUB_TOKEN")
assert BASE.startswith("https://") and "your-tunnel" not in BASE, "请设置 BASE_URL Kaggle Secret"
assert TOKEN, "请设置 KAGGLE_HUB_TOKEN Kaggle Secret"

MODEL = "triposr"
WORKER_ID = globals().get("WORKER_ID", f"triposr-{uuid.uuid4().hex[:8]}")
MODEL_DIR = Path("/kaggle/working/models/TripoSR")
MODEL_PATH = snapshot_download(
    repo_id="stabilityai/TripoSR",
    revision="5b521936b01fbe1890f6f9baed0254ab6351c04a",
    local_dir=MODEL_DIR,
    max_workers=8,
)
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
assert len(GPU_NAMES) >= 2, f"需要 2 张 GPU，当前只有 {len(GPU_NAMES)} 张"
print("worker:", WORKER_ID)
print("gpu:", GPU_NAMES[:2])
print("model:", MODEL_PATH)

## 3. 双卡加载 TripoSR

In [ ]:
def load_model(gpu):
    device = f"cuda:{gpu}"
    model = TSR.from_pretrained(MODEL_PATH, config_name="config.yaml", weight_name="model.ckpt")
    model.renderer.set_chunk_size(8192)
    model.to(device)
    model.eval()
    print(f"GPU{gpu} ready · {torch.cuda.get_device_name(gpu)} · {torch.cuda.memory_allocated(gpu)/1024**3:.2f} GiB")
    return model

MODELS = [load_model(0), load_model(1)]
REMBG_SESSION = rembg.new_session()
print("background removal ready")

## 4. Remote Dispatcher

In [ ]:
CLAIM_URL = f"{BASE}/task/next"
REGISTER_URL = f"{BASE}/worker/register"
HEARTBEAT_URL = f"{BASE}/worker/heartbeat"
FAIL_URL = f"{BASE}/task/fail"
UPLOAD_URL = f"{BASE}/upload/artifact"
KEY = hashlib.sha256(TOKEN.encode()).digest()

def auth_headers():
    return {"Authorization": f"Bearer {TOKEN}"}

def encrypt_blob(data):
    nonce = os.urandom(12)
    return nonce + AESGCM(KEY).encrypt(nonce, data, None)

def prepare_image(data, task):
    image = Image.open(io.BytesIO(data))
    image.load()
    if task.get("remove_background", True):
        image = remove_background(image, REMBG_SESSION)
        image = resize_foreground(image, float(task.get("foreground_ratio", 0.85)))
        rgba = np.asarray(image).astype(np.float32) / 255.0
        rgb = rgba[:, :, :3] * rgba[:, :, 3:4] + (1.0 - rgba[:, :, 3:4]) * 0.5
        return Image.fromarray((rgb * 255.0).astype(np.uint8))
    return image.convert("RGB")

def reconstruct(model, gpu, image, task):
    device = f"cuda:{gpu}"
    model.renderer.set_chunk_size(int(task.get("chunk_size", 8192)))
    with torch.inference_mode():
        scene_codes = model([image], device=device)
        meshes = model.extract_mesh(scene_codes, True, resolution=int(task.get("mc_resolution", 256)))
    mesh = meshes[0]
    output_format = task.get("output_format", "glb")
    with tempfile.TemporaryDirectory(prefix=f"triposr-{task['id']}-") as tmp:
        path = Path(tmp) / f"mesh.{output_format}"
        mesh.export(path)
        result = path.read_bytes()
    vertices, faces = len(mesh.vertices), len(mesh.faces)
    del scene_codes, meshes, mesh
    torch.cuda.empty_cache()
    return result, output_format, vertices, faces

class Dispatcher:
    def __init__(self, models):
        self.models = models
        self.stop_event = threading.Event()
        self.gpu_threads = []
        self.heartbeat_thread = None
        self.active = [None, None]

    def request(self, method, url, **kwargs):
        headers = dict(auth_headers())
        headers.update(kwargs.pop("headers", {}))
        return requests.request(method, url, headers=headers, **kwargs)

    def register(self):
        r = self.request("POST", REGISTER_URL, json={
            "worker_id": WORKER_ID, "model": MODEL, "gpus": GPU_NAMES[:2],
            "runtime": "official TripoSR", "concurrency": 2,
            "meta": {"commit": TRIPOSR_COMMIT, "default_mc_resolution": 256},
        }, timeout=20)
        r.raise_for_status()
        print(f"registered {WORKER_ID} -> {MODEL}")

    def heartbeat_loop(self):
        while not self.stop_event.is_set():
            try:
                r = self.request("POST", HEARTBEAT_URL, json={
                    "worker_id": WORKER_ID, "local_queue": sum(x is not None for x in self.active),
                    "upload_queue": 0, "meta": {"active_tasks": self.active},
                }, timeout=15)
                if r.status_code == 404:
                    self.register()
                else:
                    r.raise_for_status()
            except Exception as e:
                if not self.stop_event.is_set(): print("! heartbeat:", e)
            self.stop_event.wait(10)

    def claim(self, session):
        while not self.stop_event.is_set():
            try:
                r = session.get(CLAIM_URL, params={"model": MODEL, "worker_id": WORKER_ID}, timeout=(10, 35))
                if r.status_code == 204: continue
                if r.status_code == 401: self.stop_event.set(); raise RuntimeError("Hub token 无效")
                r.raise_for_status()
                return r.json()
            except requests.RequestException as e:
                if not self.stop_event.is_set(): print("! claim:", e); time.sleep(2)
        return None

    def fail(self, task_id, error, requeue=True):
        try:
            r = self.request("POST", FAIL_URL, json={"id": task_id, "error": str(error)[:2000], "requeue": requeue}, timeout=15)
            if r.status_code not in (200, 404): r.raise_for_status()
        except Exception as e: print(f"! fail report #{task_id}: {e}")

    def upload(self, task, gpu, data, output_format, vertices, faces, seconds):
        encrypted = encrypt_blob(data)
        fields = {
            "id": str(task["id"]), "model": MODEL, "worker_id": WORKER_ID,
            "gpu": str(gpu), "seconds": str(seconds), "output_format": output_format,
            "vertices": str(vertices), "faces": str(faces),
        }
        last = None
        for attempt in range(8):
            try:
                r = self.request("POST", UPLOAD_URL, data=fields, files={
                    "file": (f"mesh.{output_format}.bin", encrypted, "application/octet-stream")
                }, timeout=(10, 300))
                if r.status_code == 409: raise RuntimeError("任务租约已失效: " + r.text[:300])
                r.raise_for_status()
                return
            except Exception as e:
                last = e
                if attempt < 7: time.sleep(min(30, 2 ** attempt))
        raise last

    def gpu_loop(self, gpu):
        torch.cuda.set_device(gpu)
        session = requests.Session(); session.headers.update(auth_headers())
        while not self.stop_event.is_set():
            task = self.claim(session)
            if task is None: continue
            self.active[gpu] = task["id"]
            start = time.perf_counter()
            print(f"↓ #{task['id']:03d} | GPU{gpu} | {task.get('source_label','image')}")
            try:
                r = session.get(BASE + task["input_url"], timeout=(10, 120)); r.raise_for_status()
                image = prepare_image(r.content, task)
                data, fmt, vertices, faces = reconstruct(self.models[gpu], gpu, image, task)
                seconds = round(time.perf_counter() - start, 3)
                self.upload(task, gpu, data, fmt, vertices, faces, seconds)
                print(f"↑ #{task['id']:03d} | GPU{gpu} | {fmt.upper()} | {vertices} vertices | {seconds:.2f}s")
            except Exception as e:
                print(f"✗ #{task['id']:03d} | GPU{gpu} | {e}")
                self.fail(task["id"], e, True)
            finally:
                self.active[gpu] = None
        session.close()

    def start(self):
        self.register()
        self.heartbeat_thread = threading.Thread(target=self.heartbeat_loop, name="triposr-heartbeat", daemon=True)
        self.gpu_threads = [threading.Thread(target=self.gpu_loop, args=(gpu,), name=f"triposr-gpu{gpu}", daemon=True) for gpu in range(2)]
        self.heartbeat_thread.start()
        for thread in self.gpu_threads: thread.start()
        print("TripoSR Dispatcher ready · GPU0 + GPU1 · waiting for images...")
        return self

    def stop(self):
        self.stop_event.set()
        print("Dispatcher stop requested；模型仍保留在 GPU")

    def status(self):
        print("worker:", WORKER_ID, "| stop:", self.stop_event.is_set(), "| active:", self.active)
        for i, thread in enumerate(self.gpu_threads): print(f"gpu{i}:", thread.is_alive())

old = globals().get("DISPATCHER")
if old is not None and any(t.is_alive() for t in old.gpu_threads):
    print("Dispatcher 已在运行，不重复启动")
else:
    DISPATCHER = Dispatcher(MODELS).start()

## 5. 状态 / 可选停止

In [ ]:
DISPATCHER.status()

# 停止领取新任务（已经领取的请求可能仍会完成）：
# DISPATCHER.stop()

# 需要释放 GPU 时：
# DISPATCHER.stop()
# del MODELS
# torch.cuda.empty_cache()